In [24]:
### Document Data Structure

from langchain_core.documents import Document

doc = Document(
    page_content="This is a sample document.",
    metadata={
        "source": "sample.txt",
        "page": 1,
        "author":"Trisita Ghosh",
        "date":"2026-03-18",
    }
)

print(doc)

page_content='This is a sample document.' metadata={'source': 'sample.txt', 'page': 1, 'author': 'Trisita Ghosh', 'date': '2026-03-18'}


In [25]:
##Creating a Document

import os
os.makedirs("../data/text_files", exist_ok=True)



In [26]:
sample_text ={
    "../data/text_files/python_intro.txt": """
    Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python was designed with a philosophy that emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code than might be used in languages such as C++ or Java.

    Key Features:
    - Easy to Learn and Use: Python's syntax is clean and intuitive, making it an excellent choice for beginners.
    - Dynamically Typed: You don't need to declare variable types explicitly; Python handles this automatically.
    - Large Standard Library: It comes with a vast collection of pre-written code modules that can be used for various tasks.
    - Versatile: Python is used in web development, data science, artificial intelligence, automation, and much more.
    """,
    "../data/text_files/data_science_intro.txt": """
    Data Science is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines statistics, computer science, and domain expertise to analyze complex problems and make data-driven decisions.

    The Data Science Process:
    1. Data Collection: Gathering relevant data from various sources.
    2. Data Cleaning: Handling missing values, correcting errors, and standardizing data.
    3. Data Exploration: Understanding the data's characteristics and patterns.
    4. Modeling: Building predictive models using machine learning algorithms.
    5. Evaluation: Assessing the model's performance.
    6. Deployment: Putting the model into production.
    """
}

for filepath,content in sample_text.items():
    with open(filepath,"w", encoding="utf-8") as f:
        f.write(content)
    print(f"Created file: {filepath}")

Created file: ../data/text_files/python_intro.txt
Created file: ../data/text_files/data_science_intro.txt


In [27]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")

documents = loader.load()
# print(f"Loaded {len(documents)} documents")
print(documents)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="\n    Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python was designed with a philosophy that emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code than might be used in languages such as C++ or Java.\n\n    Key Features:\n    - Easy to Learn and Use: Python's syntax is clean and intuitive, making it an excellent choice for beginners.\n    - Dynamically Typed: You don't need to declare variable types explicitly; Python handles this automatically.\n    - Large Standard Library: It comes with a vast collection of pre-written code modules that can be used for various tasks.\n    - Versatile: Python is used in web development, data science, artificial intelligence, automation, and much more.\n    ")]


In [28]:
### Directory loader

from langchain_community.document_loaders import DirectoryLoader

##load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",  ## pattern to match files
    show_progress=False,
    #use_multithreading=True,
    loader_kwargs={
        "encoding": "utf-8"
    },
    loader_cls=TextLoader ##Loader class to use

    )

documents= dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\data_science_intro.txt'}, page_content="\n    Data Science is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines statistics, computer science, and domain expertise to analyze complex problems and make data-driven decisions.\n\n    The Data Science Process:\n    1. Data Collection: Gathering relevant data from various sources.\n    2. Data Cleaning: Handling missing values, correcting errors, and standardizing data.\n    3. Data Exploration: Understanding the data's characteristics and patterns.\n    4. Modeling: Building predictive models using machine learning algorithms.\n    5. Evaluation: Assessing the model's performance.\n    6. Deployment: Putting the model into production.\n    "),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content="\n    Python is a high-level, inter

In [29]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

##load pdf files
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",  ## pattern to match files
    show_progress=False,
    loader_cls=PyMuPDFLoader ##Loader class to use

    )

pdf_documents= dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-18T11:20:51+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-18T11:20:51+00:00', 'trapped': '', 'modDate': "D:20260318112051+00'00'", 'creationDate': "D:20260318112051+00'00'", 'page': 0}, page_content="Attention heads are a fundamental component of the transformer architecture, widely used in\nmodern deep learning models such as GPT and BERT. An attention head allows the model to focus\non different parts of an input sequence when making predictions. Instead of processing all words\nequally, the attention mechanism assigns different importance (weights) to different words based on\ntheir relevance to a specific task. Each attention head operates independently and lear

In [30]:
### Chunking with correct import

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

# For text files
chunks = splitter.split_documents(documents)

# For PDF files
pdf_chunks = splitter.split_documents(pdf_documents)

print(f"Text chunks: {len(chunks)}")
print(f"PDF chunks: {len(pdf_chunks)}")


Text chunks: 4
PDF chunks: 16


Embedding and VectorStoreDB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer and stores them in ChromaDB  """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding Manager

        Args:
            model_name (str): HuggingFace model name for sentence embeddings
        """
        self.model_name= model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ... ")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


##Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading model: all-MiniLM-L6-v2


c:\Users\trish\OneDrive\Desktop\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\trish\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1507.75it/s]
BertModel LOAD REPO

Model loaded successfully


VectorStore

In [32]:
class VectorStore:
    """Manages document embedding in a ChromaDb vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str= "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None 
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the vector store"""
        try:
            # Create persistent ChromeDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata= {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized successfully. Collection:{self.collection_name}")
            print(f"Existing documents count: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {str(e)}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must have the same length")
        print(f"Adding {len(documents)} documents to vector store")

        #prepare data for chromadb

        ids=[]
        metadatas =[]
        documents_text =[]
        embeddings_list =[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata= dict (doc.metadata)
            metadata['doc_index'] = i 
            metadata['content_length'] =len(doc.page_content)
            metadatas.append(metadata)

            # Document Content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())
        
        #add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(ids)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents to vector store: {str(e)}")
            raise

vectorstore= VectorStore()
vectorstore

Vector store initialized successfully. Collection:pdf_documents
Existing documents count: 0


In [33]:
pdf_chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-18T11:20:51+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-18T11:20:51+00:00', 'trapped': '', 'modDate': "D:20260318112051+00'00'", 'creationDate': "D:20260318112051+00'00'", 'page': 0}, page_content='Attention heads are a fundamental component of the transformer architecture, widely used in\nmodern deep learning models such as GPT and BERT. An attention head allows the model to focus\non different parts of an input sequence when making predictions. Instead of processing all words\nequally, the attention mechanism assigns different importance (weights) to different words based on\ntheir relevance to a specific task. Each attention head operates independently and lear

In [34]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\data_science_intro.txt'}, page_content='Data Science is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines statistics, computer science, and domain expertise to analyze complex problems and make data-driven decisions.'),
 Document(metadata={'source': '..\\data\\text_files\\data_science_intro.txt'}, page_content="The Data Science Process:\n    1. Data Collection: Gathering relevant data from various sources.\n    2. Data Cleaning: Handling missing values, correcting errors, and standardizing data.\n    3. Data Exploration: Understanding the data's characteristics and patterns.\n    4. Modeling: Building predictive models using machine learning algorithms.\n    5. Evaluation: Assessing the model's performance.\n    6. Deployment: Putting the model into production."),
 Document(metadata={'source': '..\\data\\text

In [37]:
#convert the text to embedding
texts = [doc.page_content for doc in chunks]
#generate the embedding
embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks , embeddings)

Generating embeddings for 4 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


Generated embeddings with shape: (4, 384)
Adding 4 documents to vector store
Successfully added 4 documents to vector store


In [38]:
#convert the pdf to embedding
texts = [doc.page_content for doc in pdf_chunks]
#generate the embedding
embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(pdf_chunks , embeddings)

Generating embeddings for 16 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]

Generated embeddings with shape: (16, 384)
Adding 16 documents to vector store
Successfully added 16 documents to vector store


Retriever Pipeline From VectorStore

In [45]:
from typing import List, Dict, Any

class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                distances = results['distances'][0]
                metadatas = results['metadatas'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'document': document,
                            'metadata': metadata,
                            'distance': distance,
                            'similarity_score': similarity_score,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [47]:
rag_retriever.retrieve("What is python?")

Retrieving documents for query: 'What is python?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.40it/s]

Generated embeddings with shape: (1, 384)


Retrieved 2 documents


[{'id': 'doc_9cbdfbf3_2',
  'document': 'Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python was designed with a philosophy that emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code than might be used in languages such as C++ or Java.',
  'metadata': {'source': '..\\data\\text_files\\python_intro.txt',
   'content_length': 358,
   'doc_index': 2},
  'distance': 0.3809467852115631,
  'similarity_score': 0.6190532147884369,
  'rank': 1},
 {'id': 'doc_ef4464db_3',
  'document': "Key Features:\n    - Easy to Learn and Use: Python's syntax is clean and intuitive, making it an excellent choice for beginners.\n    - Dynamically Typed: You don't need to declare variable types explicitly; Python handles this automatically.\n    - Large Standard Library: It comes with a vast collection of pre-written code modules that can be used for

Integration Vectordb Context pipeline with LLM output

In [53]:
# Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",  
    temperature=0.1,
    max_tokens=1024
)

# Simple RAG function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc['document'] for doc in results]) if results else ""

    if not context:
        return "No context found for the query."

    # Generate answer using Groq LLM
    prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:"""

    response = llm.invoke(prompt)

    return response.content

In [56]:
answer=rag_simple("What is data science?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is data science?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.74it/s]

Generated embeddings with shape: (1, 384)


Retrieved 2 documents
Data Science is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines statistics, computer science, and domain expertise to analyze complex problems and make data-driven decisions.
